# Interpretable Telematics-Based Energy Anomaly Detection — final notebook

This notebook is **end-to-end and reproducible**:

1. builds a per-trip table from raw VED files,
2. trains and tunes an **XGBoost regressor**,
3. detects anomalies through **positive residuals**,
4. implements **LIME for tabular data from scratch**,
5. saves final artifacts and plots.

The notebook does **not** use any ready-made XAI libraries.


## 0. What you need to fill once

Only these paths must be set manually:

- `WEEK_FILES_GLOB`
- `STATIC_PHEV_EV_XLSX`
- `STATIC_ICE_HEV_XLSX`

Everything else is deterministic and saved to `outputs_final/`.


In [ ]:

from pathlib import Path
import glob
import json
import math
import warnings
from dataclasses import dataclass, asdict
from itertools import product

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.options.display.float_format = lambda x: f"{x:.4f}"

# ============================================================
# TODO: fill these three paths manually before the first run
# ============================================================
WEEK_FILES_GLOB = r"TODO/VED_DynamicData_Part*/VED_*_week.csv"
STATIC_PHEV_EV_XLSX = Path(r"TODO/VED_Static_Data_PHEV_EV.xlsx")
STATIC_ICE_HEV_XLSX = Path(r"TODO/VED_Static_Data_ICE_HEV.xlsx")

# ============================================================
# Output / cache
# ============================================================
PROJECT_ROOT = Path(".").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs_final"
CACHE_DIR = OUTPUT_DIR / "cache"
MODEL_DIR = OUTPUT_DIR / "models"
PLOT_DIR = OUTPUT_DIR / "plots"
EXPLAIN_DIR = OUTPUT_DIR / "lime_explanations"

for p in [OUTPUT_DIR, CACHE_DIR, MODEL_DIR, PLOT_DIR, EXPLAIN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRIP_TABLE_CACHE = CACHE_DIR / "trip_table_raw.csv.gz"
SCORED_TRIP_TABLE_PATH = CACHE_DIR / "trip_table_scored.csv.gz"
MODEL_ARTIFACT_PATH = MODEL_DIR / "xgb_energy_artifact.joblib"
XGB_TUNING_PATH = CACHE_DIR / "xgb_tuning_results.csv"
LIME_TUNING_PATH = CACHE_DIR / "lime_tuning_results.csv"
EXPLANATION_SUMMARY_PATH = EXPLAIN_DIR / "lime_explanation_summary.csv"

USE_CACHED_TRIP_TABLE = True
USE_CACHED_SCORED_TABLE = False
RUN_XGB_TUNING = True
RUN_LIME_TUNING = True

# ============================================================
# Model / anomaly / explanation settings
# ============================================================
TARGET = "energy_per_km"
ID_COLS = ["trip_id", "VehId", "Trip"]

NUMERIC_FEATURES = [
    "duration_min",
    "distance_km",
    "speed_mean",
    "speed_var",
    "accel_mean",
    "accel_var",
    "accel_p95",
    "stop_go_ratio",
    "idle_time_min",
    "Generalized_Weight",
]

CATEGORICAL_FEATURES = [
    "VehicleType",
    "Vehicle Class",
    "Transmission",
    "Drive Wheels",
]

RAW_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

ANOMALY_QUANTILE = 0.98
N_EXPLANATIONS = 5
TOP_K_FEATURES = 10
BACKGROUND_SAMPLE_SIZE = 4000

# ============================================================
# VED column names used in the dynamic files
# ============================================================
COL_VEHID = "VehId"
COL_TRIP = "Trip"
COL_TS_MS = "Timestamp(ms)"
COL_SPEED = "Vehicle Speed[km/h]"
COL_FUEL_RATE = "Fuel Rate[L/hr]"

COL_AC_KW = "Air Conditioning Power[kW]"
COL_AC_W = "Air Conditioning Power[Watts]"
COL_HEATER_W = "Heater Power[Watts]"
COL_HV_I = "HV Battery Current[A]"
COL_HV_V = "HV Battery Voltage[V]"

GASOLINE_KWH_PER_LITER = 8.9
SPEED_STOP_THRESHOLD = 5.0

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_DIR   =", OUTPUT_DIR)


## 1. Utilities: ingestion, trip aggregation, static merge

In [ ]:

def list_week_files(week_files_glob: str) -> list[Path]:
    files = sorted(Path(p) for p in glob.glob(week_files_glob))
    if not files:
        raise FileNotFoundError(
            "No week files matched WEEK_FILES_GLOB. "
            "Fill the TODO path in the config cell first."
        )
    return files


def compute_instant_power_kW(df: pd.DataFrame) -> np.ndarray:
    # Fuel power
    if COL_FUEL_RATE in df.columns:
        fuel = pd.to_numeric(df[COL_FUEL_RATE], errors="coerce").fillna(0.0).to_numpy()
    else:
        fuel = np.zeros(len(df), dtype=float)
    p_fuel = fuel * GASOLINE_KWH_PER_LITER

    # Battery power
    if (COL_HV_I in df.columns) and (COL_HV_V in df.columns):
        i = pd.to_numeric(df[COL_HV_I], errors="coerce").fillna(0.0).to_numpy()
        v = pd.to_numeric(df[COL_HV_V], errors="coerce").fillna(0.0).to_numpy()
        p_batt = (v * i) / 1000.0
    else:
        p_batt = np.zeros(len(df), dtype=float)

    # HVAC
    ac = np.zeros(len(df), dtype=float)
    if COL_AC_KW in df.columns:
        ac += pd.to_numeric(df[COL_AC_KW], errors="coerce").fillna(0.0).to_numpy()
    elif COL_AC_W in df.columns:
        ac += pd.to_numeric(df[COL_AC_W], errors="coerce").fillna(0.0).to_numpy() / 1000.0

    heater = np.zeros(len(df), dtype=float)
    if COL_HEATER_W in df.columns:
        heater += pd.to_numeric(df[COL_HEATER_W], errors="coerce").fillna(0.0).to_numpy() / 1000.0

    return p_fuel + p_batt + ac + heater


def integrate_trip_energy_distance(trip_df: pd.DataFrame) -> tuple[float, float, float]:
    trip_df = trip_df.sort_values(COL_TS_MS)
    t = pd.to_numeric(trip_df[COL_TS_MS], errors="coerce").to_numpy() / 1000.0
    if len(t) < 2:
        return 0.0, 0.0, 0.0

    dt = np.diff(t)
    dt = np.clip(dt, 0.0, None)

    speed_kmh = pd.to_numeric(trip_df[COL_SPEED], errors="coerce").fillna(0.0).to_numpy()
    speed_kms = speed_kmh / 3600.0
    p_kW = compute_instant_power_kW(trip_df)

    energy_kWh = float(np.sum(p_kW[:-1] * (dt / 3600.0)))
    distance_km = float(np.sum(speed_kms[:-1] * dt))
    duration_min = float((t[-1] - t[0]) / 60.0)
    return energy_kWh, distance_km, duration_min


def compute_trip_features(trip_df: pd.DataFrame) -> dict:
    trip_df = trip_df.sort_values(COL_TS_MS)
    t = pd.to_numeric(trip_df[COL_TS_MS], errors="coerce").to_numpy() / 1000.0
    if len(t) < 2:
        return {}

    dt = np.diff(t)
    dt = np.clip(dt, 0.0, None)
    dt_sum = float(np.sum(dt)) + 1e-12

    speed_kmh = pd.to_numeric(trip_df[COL_SPEED], errors="coerce").fillna(0.0).to_numpy()
    speed_ms = speed_kmh * (1000.0 / 3600.0)

    w = dt / dt_sum
    speed_mean = float(np.sum(speed_kmh[:-1] * w))
    speed_var = float(np.sum(((speed_kmh[:-1] - speed_mean) ** 2) * w))

    accel = np.diff(speed_ms) / (dt + 1e-12)
    accel = np.nan_to_num(accel, nan=0.0, posinf=0.0, neginf=0.0)
    accel_mean = float(np.sum(accel * w))
    accel_var = float(np.sum(((accel - accel_mean) ** 2) * w))
    accel_p95 = float(np.percentile(accel, 95))

    stop_go_ratio = float(np.mean(speed_kmh < SPEED_STOP_THRESHOLD))
    idle_time_min = float(np.sum(dt[speed_kmh[:-1] < 0.1]) / 60.0)

    energy_kWh, distance_km, duration_min = integrate_trip_energy_distance(trip_df)
    energy_per_km = np.nan if distance_km <= 1e-6 else energy_kWh / distance_km

    return {
        "duration_min": duration_min,
        "distance_km": distance_km,
        "speed_mean": speed_mean,
        "speed_var": speed_var,
        "accel_mean": accel_mean,
        "accel_var": accel_var,
        "accel_p95": accel_p95,
        "stop_go_ratio": stop_go_ratio,
        "idle_time_min": idle_time_min,
        "energy_kWh": energy_kWh,
        "energy_per_km": energy_per_km,
    }


def aggregate_week_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["trip_id"] = df[COL_VEHID].astype(str) + "_" + df[COL_TRIP].astype(str)

    rows = []
    for trip_id, g in df.groupby("trip_id", sort=False):
        feat = compute_trip_features(g)
        if not feat:
            continue
        feat["trip_id"] = trip_id
        feat["VehId"] = int(g[COL_VEHID].iloc[0])
        feat["Trip"] = int(g[COL_TRIP].iloc[0])
        rows.append(feat)

    return pd.DataFrame(rows)


def load_static_tables(phev_ev_path: Path, ice_hev_path: Path) -> pd.DataFrame:
    if not phev_ev_path.exists():
        raise FileNotFoundError(f"Missing static file: {phev_ev_path}")
    if not ice_hev_path.exists():
        raise FileNotFoundError(f"Missing static file: {ice_hev_path}")

    phev = pd.read_excel(phev_ev_path)
    ice = pd.read_excel(ice_hev_path)

    phev = phev.rename(columns={"EngineType": "VehicleType"})
    ice = ice.rename(columns={"Vehicle Type": "VehicleType"})

    static = pd.concat([phev, ice], ignore_index=True)
    static = static.drop_duplicates(subset=["VehId"]).reset_index(drop=True)
    return static


def prepare_trip_table(
    week_files_glob: str,
    phev_ev_path: Path,
    ice_hev_path: Path,
    cache_path: Path | None = None,
    use_cache: bool = True,
) -> pd.DataFrame:
    if use_cache and cache_path is not None and cache_path.exists():
        trips = pd.read_csv(cache_path)
        return trips

    week_files = list_week_files(week_files_glob)
    print(f"Found {len(week_files)} dynamic files")

    trip_tables = []
    for i, f in enumerate(week_files, start=1):
        tdf = aggregate_week_file(f)
        trip_tables.append(tdf)
        if i % 5 == 0 or i == len(week_files):
            trips_so_far = sum(len(x) for x in trip_tables)
            print(f"Processed {i}/{len(week_files)} files | trips so far = {trips_so_far}")

    trips = pd.concat(trip_tables, ignore_index=True)
    trips = trips.dropna(subset=[TARGET])
    trips = trips[trips["distance_km"] > 0.1].reset_index(drop=True)

    static_df = load_static_tables(phev_ev_path, ice_hev_path)
    trips = trips.merge(static_df, on="VehId", how="left")

    trips["Generalized_Weight"] = pd.to_numeric(trips["Generalized_Weight"], errors="coerce")

    for col in CATEGORICAL_FEATURES:
        trips[col] = trips[col].astype("string").fillna("NO DATA")

    for col in NUMERIC_FEATURES:
        trips[col] = pd.to_numeric(trips[col], errors="coerce")

    if cache_path is not None:
        trips.to_csv(cache_path, index=False, compression="gzip")

    return trips


def split_by_vehicle(trips: pd.DataFrame, test_size: float = 0.2, val_size: float = 0.2, seed: int = 42):
    vehicles = trips["VehId"].dropna().unique()
    train_val, test = train_test_split(vehicles, test_size=test_size, random_state=seed)
    train, val = train_test_split(train_val, test_size=val_size / (1.0 - test_size), random_state=seed)
    return (
        trips["VehId"].isin(train),
        trips["VehId"].isin(val),
        trips["VehId"].isin(test),
    )


## 2. Build the raw trip table (cached after the first run)

In [ ]:

trips_df = prepare_trip_table(
    week_files_glob=WEEK_FILES_GLOB,
    phev_ev_path=STATIC_PHEV_EV_XLSX,
    ice_hev_path=STATIC_ICE_HEV_XLSX,
    cache_path=TRIP_TABLE_CACHE,
    use_cache=USE_CACHED_TRIP_TABLE,
)

print("trip table shape:", trips_df.shape)
display(trips_df[ID_COLS + RAW_FEATURES + [TARGET]].head())


## 3. Train / validation / test split by vehicle id

In [ ]:

train_mask, val_mask, test_mask = split_by_vehicle(trips_df, seed=RANDOM_STATE)

trips_df = trips_df.copy()
trips_df["split"] = np.select(
    [train_mask, val_mask, test_mask],
    ["train", "val", "test"],
    default="unknown",
)

train_df = trips_df[trips_df["split"] == "train"].reset_index(drop=True)
val_df = trips_df[trips_df["split"] == "val"].reset_index(drop=True)
test_df = trips_df[trips_df["split"] == "test"].reset_index(drop=True)

print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)


## 4. Design matrix for XGBoost

In [ ]:

def fill_raw_features(df: pd.DataFrame, numeric_fill: dict[str, float] | None = None) -> pd.DataFrame:
    out = df.copy()

    if numeric_fill is None:
        numeric_fill = {
            col: float(pd.to_numeric(out[col], errors="coerce").median())
            for col in NUMERIC_FEATURES
        }

    for col in NUMERIC_FEATURES:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(numeric_fill[col])

    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].astype("string").fillna("NO DATA")

    return out


def make_design(
    df: pd.DataFrame,
    design_columns: list[str] | None = None,
    numeric_fill: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, dict[str, float], list[str]]:
    raw = fill_raw_features(df[RAW_FEATURES], numeric_fill=numeric_fill)
    X = pd.get_dummies(raw, columns=CATEGORICAL_FEATURES, dummy_na=False)

    if design_columns is None:
        design_columns = list(X.columns)
    X = X.reindex(columns=design_columns, fill_value=0.0)

    if numeric_fill is None:
        numeric_fill = {col: float(raw[col].median()) for col in NUMERIC_FEATURES}

    return X.astype(float), numeric_fill, list(design_columns)


y_train = train_df[TARGET].astype(float).to_numpy()
y_val = val_df[TARGET].astype(float).to_numpy()
y_test = test_df[TARGET].astype(float).to_numpy()

X_train, numeric_fill_values, design_columns_train = make_design(train_df)
X_val, _, _ = make_design(val_df, design_columns=design_columns_train, numeric_fill=numeric_fill_values)
X_test, _, _ = make_design(test_df, design_columns=design_columns_train, numeric_fill=numeric_fill_values)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


## 5. Tune XGBoost on validation RMSE

In [ ]:

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_regression(y_true, y_pred) -> dict:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "r2": float(r2_score(y_true, y_pred)),
    }


def tune_xgb_regressor(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    random_state: int = 42,
) -> pd.DataFrame:
    param_grid = {
        "max_depth": [4, 6],
        "learning_rate": [0.03, 0.05],
        "min_child_weight": [1, 5],
        "reg_lambda": [1.0, 3.0],
    }

    common = {
        "n_estimators": 1500,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": random_state,
        "early_stopping_rounds": 50,
        "eval_metric": "rmse",
    }

    rows = []
    all_keys = list(param_grid.keys())
    for values in product(*[param_grid[k] for k in all_keys]):
        params = dict(zip(all_keys, values))
        model = XGBRegressor(**common, **params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )

        val_pred = model.predict(X_val)
        metrics = evaluate_regression(y_val, val_pred)

        best_iteration = getattr(model, "best_iteration", None)
        best_n_estimators = common["n_estimators"] if best_iteration is None else int(best_iteration) + 1

        row = {**params, **metrics, "best_n_estimators": best_n_estimators}
        rows.append(row)

    result = pd.DataFrame(rows).sort_values(["rmse", "mae"], ascending=[True, True]).reset_index(drop=True)
    return result


if RUN_XGB_TUNING or not XGB_TUNING_PATH.exists():
    xgb_tuning_df = tune_xgb_regressor(X_train, y_train, X_val, y_val, random_state=RANDOM_STATE)
    xgb_tuning_df.to_csv(XGB_TUNING_PATH, index=False)
else:
    xgb_tuning_df = pd.read_csv(XGB_TUNING_PATH)

display(xgb_tuning_df.head(10))


## 6. Fit the final XGBoost model on train + val using the selected number of trees

In [ ]:

best_xgb_row = xgb_tuning_df.iloc[0].to_dict()
best_xgb_params = {
    "max_depth": int(best_xgb_row["max_depth"]),
    "learning_rate": float(best_xgb_row["learning_rate"]),
    "min_child_weight": int(best_xgb_row["min_child_weight"]),
    "reg_lambda": float(best_xgb_row["reg_lambda"]),
    "n_estimators": int(best_xgb_row["best_n_estimators"]),
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "eval_metric": "rmse",
}

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
y_train_val = train_val_df[TARGET].astype(float).to_numpy()
X_train_val, numeric_fill_values_final, design_columns_final = make_design(train_val_df)

final_model = XGBRegressor(**best_xgb_params)
final_model.fit(X_train_val, y_train_val, verbose=False)

X_test_final, _, _ = make_design(
    test_df,
    design_columns=design_columns_final,
    numeric_fill=numeric_fill_values_final,
)

test_pred = final_model.predict(X_test_final)
test_metrics = evaluate_regression(y_test, test_pred)

print("Selected XGBoost params:")
print(json.dumps(best_xgb_params, indent=2))
print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))


## 7. Score all trips and save the final table with raw features + residuals

This is the important fix missing from the previous pipeline:  
for LIME we need the **raw feature row** of every explained trip, not only residuals.


In [ ]:

def predict_on_raw(df_raw: pd.DataFrame, model: XGBRegressor, design_columns: list[str], numeric_fill: dict[str, float]) -> np.ndarray:
    X, _, _ = make_design(df_raw, design_columns=design_columns, numeric_fill=numeric_fill)
    return model.predict(X)


scored_df = trips_df.copy()
scored_df["predicted_energy_per_km"] = predict_on_raw(
    scored_df,
    model=final_model,
    design_columns=design_columns_final,
    numeric_fill=numeric_fill_values_final,
)
scored_df["residual"] = scored_df[TARGET] - scored_df["predicted_energy_per_km"]

scored_df.to_csv(SCORED_TRIP_TABLE_PATH, index=False, compression="gzip")

artifact = {
    "model": final_model,
    "design_columns": design_columns_final,
    "numeric_fill_values": numeric_fill_values_final,
    "raw_feature_columns": RAW_FEATURES,
    "numeric_feature_columns": NUMERIC_FEATURES,
    "categorical_feature_columns": CATEGORICAL_FEATURES,
    "background_raw": train_val_df[RAW_FEATURES].sample(
        n=min(BACKGROUND_SAMPLE_SIZE, len(train_val_df)),
        random_state=RANDOM_STATE,
    ).reset_index(drop=True),
    "best_xgb_params": best_xgb_params,
    "test_metrics": test_metrics,
}

joblib.dump(artifact, MODEL_ARTIFACT_PATH)

print("Saved scored trip table:", SCORED_TRIP_TABLE_PATH)
print("Saved model artifact   :", MODEL_ARTIFACT_PATH)

display(scored_df[ID_COLS + RAW_FEATURES + [TARGET, "predicted_energy_per_km", "residual", "split"]].head())


## 8. Basic diagnostics

In [ ]:

fig = plt.figure(figsize=(6, 5))
plt.scatter(
    scored_df[TARGET],
    scored_df["predicted_energy_per_km"],
    s=10,
    alpha=0.35,
)
vmin = float(min(scored_df[TARGET].min(), scored_df["predicted_energy_per_km"].min()))
vmax = float(max(scored_df[TARGET].max(), scored_df["predicted_energy_per_km"].max()))
plt.plot([vmin, vmax], [vmin, vmax], linestyle="--")
plt.xlabel("Actual energy_per_km")
plt.ylabel("Predicted energy_per_km")
plt.title("Prediction quality")
plt.tight_layout()
plt.savefig(PLOT_DIR / "predicted_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

fig = plt.figure(figsize=(7, 4))
plt.hist(scored_df["residual"], bins=80)
plt.axvline(0.0, linestyle="--")
plt.xlabel("Residual = actual - predicted")
plt.ylabel("Count")
plt.title("Residual distribution")
plt.tight_layout()
plt.savefig(PLOT_DIR / "residual_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Residual-based anomaly detection

In [ ]:

def positive_quantile_threshold(residuals: pd.Series, q: float = 0.98) -> float:
    if not (0.0 < q < 1.0):
        raise ValueError("q must be in (0, 1)")
    return float(np.quantile(residuals.astype(float), q))


residual_threshold = positive_quantile_threshold(scored_df["residual"], q=ANOMALY_QUANTILE)
scored_df["is_anomaly"] = scored_df["residual"] >= residual_threshold

n_anomalies = int(scored_df["is_anomaly"].sum())
share_anomalies = float(scored_df["is_anomaly"].mean())

print("Residual threshold:", residual_threshold)
print("Anomalies         :", n_anomalies)
print("Share             :", share_anomalies)

display(
    scored_df.loc[scored_df["is_anomaly"], ID_COLS + RAW_FEATURES + [TARGET, "predicted_energy_per_km", "residual"]]
    .sort_values("residual", ascending=False)
    .head(10)
)


## 10. Manual LIME implementation from scratch

Implementation choices:

- perturbation in **raw feature space**,
- categorical features sampled from the empirical training distribution,
- numerical features sampled from a Gaussian around the explained point and clipped to the training range,
- distances computed in an **interpretable standardized space**,
- local surrogate = **weighted ridge regression in closed form**,
- no `lime` / `shap` / other XAI libraries.


In [ ]:

@dataclass
class LimeExplanation:
    trip_id: str
    blackbox_prediction: float
    surrogate_prediction_at_x0: float
    intercept: float
    kernel_width: float
    local_r2: float
    local_rmse: float
    top_features: list[tuple[str, float]]


def sample_background_stats(background_raw: pd.DataFrame):
    stats = {}
    for col in NUMERIC_FEATURES:
        x = pd.to_numeric(background_raw[col], errors="coerce").astype(float)
        x = x.dropna()
        sigma = float(x.std(ddof=0))
        if not np.isfinite(sigma) or sigma < 1e-12:
            sigma = 1.0
        stats[col] = {
            "mean": float(x.mean()),
            "std": sigma,
            "q01": float(x.quantile(0.01)),
            "q99": float(x.quantile(0.99)),
        }
    cat_probs = {}
    for col in CATEGORICAL_FEATURES:
        vc = background_raw[col].astype("string").fillna("NO DATA").value_counts(normalize=True)
        cat_probs[col] = {
            "values": vc.index.astype(str).tolist(),
            "probs": vc.values.astype(float),
        }
    return stats, cat_probs


def perturb_raw_instance(
    x0_raw: pd.Series,
    background_raw: pd.DataFrame,
    n_samples: int,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    num_stats, cat_probs = sample_background_stats(background_raw)

    Z = pd.DataFrame(index=range(n_samples), columns=RAW_FEATURES)

    for col in NUMERIC_FEATURES:
        s = num_stats[col]
        loc = float(x0_raw[col])
        z = rng.normal(loc=loc, scale=s["std"], size=n_samples)
        z = np.clip(z, s["q01"], s["q99"])
        Z[col] = z.astype(float)

    for col in CATEGORICAL_FEATURES:
        values = np.array(cat_probs[col]["values"], dtype=object)
        probs = np.array(cat_probs[col]["probs"], dtype=float)
        Z[col] = rng.choice(values, size=n_samples, replace=True, p=probs)

    Z.iloc[0] = x0_raw[RAW_FEATURES].copy()
    return Z


def make_interpretable_matrix(
    df_raw: pd.DataFrame,
    background_raw: pd.DataFrame,
) -> pd.DataFrame:
    num_stats, _ = sample_background_stats(background_raw)

    parts = []

    num_df = pd.DataFrame(index=df_raw.index)
    for col in NUMERIC_FEATURES:
        mean_ = num_stats[col]["mean"]
        std_ = num_stats[col]["std"]
        num_df[col] = (pd.to_numeric(df_raw[col], errors="coerce").astype(float) - mean_) / std_
    parts.append(num_df)

    for col in CATEGORICAL_FEATURES:
        bg_levels = background_raw[col].astype("string").fillna("NO DATA").astype(str)
        df_levels = df_raw[col].astype("string").fillna("NO DATA").astype(str)
        categories = sorted(set(bg_levels.unique()) | set(df_levels.unique()))
        s = pd.Categorical(df_levels, categories=categories)
        dummies = pd.get_dummies(s, prefix=col, prefix_sep="=", dtype=float)
        dummies.index = df_raw.index
        parts.append(dummies)

    out = pd.concat(parts, axis=1).astype(float)
    return out


def pairwise_distance_to_x0(Z_interp: np.ndarray, x0_interp: np.ndarray) -> np.ndarray:
    return np.sqrt(np.sum((Z_interp - x0_interp[None, :]) ** 2, axis=1))


def kernel_weights(distances: np.ndarray, kernel_width: float) -> np.ndarray:
    return np.exp(-(distances ** 2) / (kernel_width ** 2 + 1e-12))


def weighted_ridge_closed_form(X: np.ndarray, y: np.ndarray, w: np.ndarray, alpha: float) -> tuple[np.ndarray, float]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    n, d = X.shape
    X1 = np.concatenate([np.ones((n, 1)), X], axis=1)

    sqrt_w = np.sqrt(w + 1e-12)
    Xw = X1 * sqrt_w[:, None]
    yw = y * sqrt_w

    ridge = np.zeros((d + 1, d + 1), dtype=float)
    ridge[1:, 1:] = alpha * np.eye(d)

    theta = np.linalg.pinv(Xw.T @ Xw + ridge) @ (Xw.T @ yw)
    intercept = float(theta[0])
    beta = theta[1:]
    return beta, intercept


def weighted_r2(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)

    y_bar = float(np.sum(w * y_true) / (np.sum(w) + 1e-12))
    ss_res = float(np.sum(w * (y_true - y_pred) ** 2))
    ss_tot = float(np.sum(w * (y_true - y_bar) ** 2)) + 1e-12
    return 1.0 - ss_res / ss_tot


def weighted_rmse(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)
    mse = float(np.sum(w * (y_true - y_pred) ** 2) / (np.sum(w) + 1e-12))
    return float(np.sqrt(mse))


def explain_one_trip_with_lime(
    x0_row: pd.Series,
    background_raw: pd.DataFrame,
    model_artifact: dict,
    n_samples: int = 4000,
    kernel_width_factor: float = 1.0,
    ridge_alpha: float = 1.0,
    top_k: int = 10,
    random_state: int = 42,
) -> tuple[LimeExplanation, pd.DataFrame]:
    Z_raw = perturb_raw_instance(
        x0_raw=x0_row[RAW_FEATURES],
        background_raw=background_raw,
        n_samples=n_samples,
        random_state=random_state,
    )

    Z_design, _, _ = make_design(
        Z_raw,
        design_columns=model_artifact["design_columns"],
        numeric_fill=model_artifact["numeric_fill_values"],
    )
    y_blackbox = model_artifact["model"].predict(Z_design)

    U = make_interpretable_matrix(Z_raw, background_raw)
    u0 = make_interpretable_matrix(x0_row[RAW_FEATURES].to_frame().T, background_raw)

    U_np = U.to_numpy(dtype=float)
    u0_np = u0.to_numpy(dtype=float).reshape(-1)

    d = pairwise_distance_to_x0(U_np, u0_np)
    kernel_width = float(kernel_width_factor * np.sqrt(U_np.shape[1]))
    w = kernel_weights(d, kernel_width=kernel_width)

    beta, intercept = weighted_ridge_closed_form(U_np, y_blackbox, w=w, alpha=ridge_alpha)
    y_sur = intercept + U_np @ beta

    local_r2_value = weighted_r2(y_blackbox, y_sur, w)
    local_rmse_value = weighted_rmse(y_blackbox, y_sur, w)

    contribution_at_x0 = beta * u0_np
    feature_names = list(U.columns)

    contrib_df = pd.DataFrame({
        "feature": feature_names,
        "beta": beta,
        "x0_interpretable_value": u0_np,
        "contribution": contribution_at_x0,
    })
    contrib_df["abs_contribution"] = contrib_df["contribution"].abs()
    contrib_df = contrib_df.sort_values("abs_contribution", ascending=False).reset_index(drop=True)

    top_features = list(
        contrib_df.loc[:, ["feature", "contribution"]]
        .head(top_k)
        .itertuples(index=False, name=None)
    )

    x0_design, _, _ = make_design(
        x0_row[RAW_FEATURES].to_frame().T,
        design_columns=model_artifact["design_columns"],
        numeric_fill=model_artifact["numeric_fill_values"],
    )
    blackbox_pred_x0 = float(model_artifact["model"].predict(x0_design)[0])
    surrogate_pred_x0 = float(intercept + np.dot(u0_np, beta))

    explanation = LimeExplanation(
        trip_id=str(x0_row["trip_id"]),
        blackbox_prediction=blackbox_pred_x0,
        surrogate_prediction_at_x0=surrogate_pred_x0,
        intercept=float(intercept),
        kernel_width=kernel_width,
        local_r2=float(local_r2_value),
        local_rmse=float(local_rmse_value),
        top_features=top_features,
    )
    return explanation, contrib_df


def plot_lime_contributions(contrib_df: pd.DataFrame, title: str, save_path: Path | None = None, top_k: int = 10):
    plot_df = contrib_df.head(top_k).iloc[::-1].copy()

    fig = plt.figure(figsize=(8, max(4, 0.45 * len(plot_df))))
    plt.barh(plot_df["feature"], plot_df["contribution"])
    plt.xlabel("Contribution to local surrogate prediction")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


## 11. Tune LIME hyperparameters on a small anomaly calibration set

Criterion: maximize median **local weighted R²** of the surrogate.


In [ ]:

artifact = joblib.load(MODEL_ARTIFACT_PATH)
background_raw = artifact["background_raw"].copy()

anomaly_pool = (
    scored_df.loc[scored_df["is_anomaly"]]
    .sort_values("residual", ascending=False)
    .drop_duplicates(subset=["VehId"])
    .head(8)
    .reset_index(drop=True)
)

calibration_trip_ids = anomaly_pool["trip_id"].astype(str).tolist()
print("Calibration trip_ids:", calibration_trip_ids)

def tune_lime(
    scored_df: pd.DataFrame,
    calibration_trip_ids: list[str],
    background_raw: pd.DataFrame,
    artifact: dict,
) -> pd.DataFrame:
    grid = {
        "n_samples": [2000, 4000],
        "kernel_width_factor": [0.75, 1.0, 1.5],
        "ridge_alpha": [0.1, 1.0, 10.0],
    }

    rows = []
    for n_samples, kernel_width_factor, ridge_alpha in product(
        grid["n_samples"],
        grid["kernel_width_factor"],
        grid["ridge_alpha"],
    ):
        r2_values = []
        rmse_values = []

        for trip_id in calibration_trip_ids:
            x0 = scored_df.loc[scored_df["trip_id"].astype(str) == str(trip_id)].iloc[0]
            exp, _ = explain_one_trip_with_lime(
                x0_row=x0,
                background_raw=background_raw,
                model_artifact=artifact,
                n_samples=n_samples,
                kernel_width_factor=kernel_width_factor,
                ridge_alpha=ridge_alpha,
                top_k=TOP_K_FEATURES,
                random_state=RANDOM_STATE,
            )
            r2_values.append(exp.local_r2)
            rmse_values.append(exp.local_rmse)

        rows.append({
            "n_samples": n_samples,
            "kernel_width_factor": kernel_width_factor,
            "ridge_alpha": ridge_alpha,
            "median_local_r2": float(np.median(r2_values)),
            "mean_local_r2": float(np.mean(r2_values)),
            "median_local_rmse": float(np.median(rmse_values)),
            "mean_local_rmse": float(np.mean(rmse_values)),
        })

    return pd.DataFrame(rows).sort_values(
        ["median_local_r2", "mean_local_r2", "median_local_rmse"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


if RUN_LIME_TUNING or not LIME_TUNING_PATH.exists():
    lime_tuning_df = tune_lime(
        scored_df=scored_df,
        calibration_trip_ids=calibration_trip_ids,
        background_raw=background_raw,
        artifact=artifact,
    )
    lime_tuning_df.to_csv(LIME_TUNING_PATH, index=False)
else:
    lime_tuning_df = pd.read_csv(LIME_TUNING_PATH)

display(lime_tuning_df.head(10))


## 12. Generate the final 3–5 explanations

In [ ]:

best_lime_row = lime_tuning_df.iloc[0].to_dict()
best_lime_params = {
    "n_samples": int(best_lime_row["n_samples"]),
    "kernel_width_factor": float(best_lime_row["kernel_width_factor"]),
    "ridge_alpha": float(best_lime_row["ridge_alpha"]),
    "top_k": TOP_K_FEATURES,
    "random_state": RANDOM_STATE,
}
print(json.dumps(best_lime_params, indent=2))

explain_candidates = (
    scored_df.loc[scored_df["is_anomaly"]]
    .sort_values("residual", ascending=False)
    .drop_duplicates(subset=["VehId"])
    .head(N_EXPLANATIONS)
    .reset_index(drop=True)
)

summary_rows = []

for _, row in explain_candidates.iterrows():
    exp, contrib_df = explain_one_trip_with_lime(
        x0_row=row,
        background_raw=background_raw,
        model_artifact=artifact,
        n_samples=best_lime_params["n_samples"],
        kernel_width_factor=best_lime_params["kernel_width_factor"],
        ridge_alpha=best_lime_params["ridge_alpha"],
        top_k=best_lime_params["top_k"],
        random_state=best_lime_params["random_state"],
    )

    safe_trip_id = str(exp.trip_id).replace("/", "_")
    contrib_path = EXPLAIN_DIR / f"trip_{safe_trip_id}_contributions.csv"
    json_path = EXPLAIN_DIR / f"trip_{safe_trip_id}_explanation.json"
    plot_path = EXPLAIN_DIR / f"trip_{safe_trip_id}_bar.png"

    contrib_df.to_csv(contrib_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(asdict(exp), f, ensure_ascii=False, indent=2)

    plot_lime_contributions(
        contrib_df=contrib_df,
        title=f"LIME explanation for trip {exp.trip_id}",
        save_path=plot_path,
        top_k=TOP_K_FEATURES,
    )

    summary_rows.append({
        "trip_id": exp.trip_id,
        "VehId": row["VehId"],
        "actual_energy_per_km": float(row[TARGET]),
        "predicted_energy_per_km": float(row["predicted_energy_per_km"]),
        "residual": float(row["residual"]),
        "local_r2": exp.local_r2,
        "local_rmse": exp.local_rmse,
        "blackbox_prediction": exp.blackbox_prediction,
        "surrogate_prediction_at_x0": exp.surrogate_prediction_at_x0,
        "top_features": " | ".join([f"{name}:{value:.4f}" for name, value in exp.top_features]),
    })

explanation_summary_df = pd.DataFrame(summary_rows)
explanation_summary_df.to_csv(EXPLANATION_SUMMARY_PATH, index=False)

display(explanation_summary_df)


## 13. Final files produced by the notebook

After a full run you will have:

- `outputs_final/cache/trip_table_raw.csv.gz`
- `outputs_final/cache/trip_table_scored.csv.gz`
- `outputs_final/cache/xgb_tuning_results.csv`
- `outputs_final/cache/lime_tuning_results.csv`
- `outputs_final/models/xgb_energy_artifact.joblib`
- `outputs_final/plots/*.png`
- `outputs_final/lime_explanations/*.json`
- `outputs_final/lime_explanations/*.csv`
- `outputs_final/lime_explanations/*.png`

This is enough for the report / demo / final presentation.
